
# Importações

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas
import random
import cudf as pd
import tensorflow as tf

from keras import Sequential
from keras.src.layers import Input, LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

from pyESN import ESN
from shap.plots import colors
from sklearn.ensemble import RandomForestRegressor as RandomForest
from sklearn.metrics import root_mean_squared_error as rmse, mean_absolute_percentage_error as mape
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from xgboost import XGBRegressor

SEED = 100
horizonte = 12
modelos = ["ESN", "LSTM", "RF", "XGB"]

def reset_seed(rnd_seed=SEED):
    os.environ['PYTHONHASHSEED'] = '0'
    random.seed(rnd_seed)
    np.random.seed(rnd_seed)
    tf.random.set_seed(rnd_seed)

reset_seed()
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/path/to/cuda'

2025-05-12 18:14:04.358324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-12 18:14:04.408647: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-12 18:14:04.433060: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-12 18:14:05.225046: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/eduardo/miniconda3/envs/rapid

# Definição das Funções
## Treino e Teste

In [ ]:
def treino_split_regr(modelo, dataframe_treino, features):
    x_treino = dataframe_treino[features].astype(np.float32).to_cupy().get()
    y_treino = dataframe_treino["CONSUMO"].astype(np.float32).to_cupy().get()

    if isinstance(modelo, RandomForest) or isinstance(modelo, XGBRegressor):
        modelo.fit(x_treino, y_treino)
    else:
        if isinstance(modelo, ESN):
            modelo.fit(x_treino, y_treino)
        else:
            modelo.fit(x_treino, y_treino, shuffle=False, verbose=False, epochs=best["LSTM"]["Epochs"],
                       batch_size=best["LSTM"]["Batch Size"])
    return modelo


def teste_split_regr(modelo, dataframe_treino, dataframe_teste, features, horizonte):
    dataframe_treino = dataframe_treino.sort_values("DATA")
    dataframe_teste = dataframe_teste.sort_values("DATA")

    dataframe_treino = dataframe_treino.set_index("DATA")
    dataframe_teste = dataframe_teste.set_index("DATA")

    xy_teste = dataframe_teste[np.append(features, "CONSUMO")].astype(np.float32)[:horizonte]
    y_teste = dataframe_teste["CONSUMO"].astype(np.float32)[:horizonte].to_cupy().get()

    dataframe_conjunto = dataframe_treino[np.append(features, "CONSUMO")].copy().astype(np.float32)

    preds = []

    for i_test in range(horizonte):
        row = xy_teste.iloc[[i_test]].copy()

        dataframe_conjunto = pd.concat([dataframe_conjunto, row], axis=0)

        for lag in range(1, 12 + 1):
            if 'LAG_' + "{:02d}".format(lag) in xy_teste.columns:
                row[f'LAG_' + '{:02d}'.format(lag)] = dataframe_conjunto["CONSUMO"].shift(lag)
                dataframe_conjunto[f'LAG_' + '{:02d}'.format(lag)] = dataframe_conjunto["CONSUMO"].shift(lag)

        if isinstance(modelo, RandomForest) or isinstance(modelo, XGBRegressor):
            pred = modelo.predict(row[features].to_cupy().get().reshape(1, -1))
        else:
            pred = modelo.predict(row[features].to_cupy().get().reshape(1, -1))[0]

        row["CONSUMO"] = pred
        preds.append(pred)
        dataframe_conjunto.update(row)

    medidas = pandas.Series([mape(y_teste, preds), rmse(y_teste, preds)], index=["MAPE", "RMSE"]).round(5)

    return medidas, dataframe_conjunto

## Criação dos Modelos

In [ ]:
def get_modelo(nome):
    if nome == "ESN":
        return ESN(n_inputs=df_features.columns.shape[0],
                   n_outputs=1,
                   n_reservoir=int(best["ESN"]["Reservoirs"]),
                   sparsity=best["ESN"]["Sparsity"],
                   spectral_radius=best["ESN"]["Spectral Radius"],
                   noise=best["ESN"]["Noise"],
                   random_state=SEED)

    if nome == "LSTM":
        tf.keras.backend.clear_session()
        lstm = Sequential([
            Input((df_features.columns.shape[0], 1)),
            LSTM(best["LSTM"]["Units"],
                 activation=best["LSTM"]["Activation"],
                 use_bias=best["LSTM"]["Bias"],
                 seed=SEED),
            Dense(1),
        ])
        lstm.compile(loss='mape')
        return lstm

    if nome == "RF":
        return RandomForest(random_state=SEED)

    if nome == "XGB":
        updater = "coord_descent" if best["XGB"]["Booster"] == "gblinear" else None
        return XGBRegressor(random_state=SEED,
                            n_estimators=int(best["XGB"]["N_estimators"]),
                            max_depth=int(best["XGB"]["Max_depth"]),
                            booster=best["XGB"]["Booster"],
                            reg_lambda=best["XGB"]["Lambda"],
                            reg_alpha=best["XGB"]["Alpha"],
                            updater=updater)


## Criação de Gráficos

In [ ]:
def ts_comparacao(campus, valor_real, valores_previstos):
    valor_real = valor_real.tail(12)

    plt.figure(figsize=(12, 4.5))
    plt.rcParams['xtick.labelsize'] = 13
    plt.rcParams['ytick.labelsize'] = 14
    plt.rcParams.update({'font.size': 12})
    plt.rcParams['axes.prop_cycle'] = plt.cycler(
        color=["blue", "green", "darkgoldenrod", colors.red_rgb, "purple", "cyan", "slategrey", "coral"])

    for modelo, previsao in valores_previstos.items():
        previsao = previsao.tail(12)
        plt.plot(previsao.index, previsao["CONSUMO"], label=f"{modelo} - {campus}")

    plt.plot(valor_real.index, valor_real["CONSUMO"], label=f"CONSUMO REAL - {campus}")

    plt.xlabel('Mês')
    plt.ylabel('Consumo (KWh)')

    ax = plt.gca()
    ax.set_facecolor('white')

    plt.xticks(valor_real.index, valor_real.index.strftime("%m/%y"))
    plt.grid(True, color='grey', linestyle="--", linewidth=0.5)
    plt.legend(facecolor='white')

    return plt



# Carregar Datasets

In [2]:
df_regressao = pandas.read_csv("./dados/dados_regressao.csv", sep=';', decimal='.')
df_features = pandas.read_csv("./resultados/features/fitness_features_regressao.csv", sep=";", decimal=".")

df_features = df_features.sort_values("MAPE").head(1).reset_index(drop=True)
df_features = pandas.DataFrame(
    columns=str(df_features.iloc[0]["FEATURES"]).replace("(", '').replace(")", '').replace("'", "").split(", "))

df_features = df_features.drop(df_features.filter(like='CAMPUS_').columns, axis=1)
df_features

,LAG_01,COVID,FÉRIAS,LAG_12,LAG_11,MÊS_mar,LAG_05,DIA_DA_SEMANA_dom,LAG_03,GREVE,...,TEMP_MAX_MAX_MENS,TEMP_MIN_MAX_MENS,ANO_2020,MÊS_fev,TEMP_MAX_MIN_MENS,DIA_DA_SEMANA_seg,MÊS_dez,ANO_2017,MÊS_nov,TEMP_MÉD_MÉD_MENS


# Melhores Parâmetros

In [3]:
best = {}
for modelo in ["ESN", "LSTM", "RF", "XGBoost"]:
    df_aux = pandas.DataFrame()
    for optimizer in ["PSO"]: # Todo remover
    # for optimizer in ["SA", "PSO"]:
        for seed in [1000]: # Todo remover
        # for seed in [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]:
            new_df = pandas.read_csv(f'resultados/otimização/{optimizer}-{modelo} SEED {seed}.csv', sep=";", decimal=".",
                                     header=0)
            df_aux = pandas.concat([df_aux, new_df], axis=0)

    df_aux = df_aux.sort_values(by=["Fitness"])
    df_aux[df_aux.isnull()] = None
    best[f"{modelo}"] = df_aux[:1].iloc[0].to_dict()

# Treino e Teste
## Divisão dos Dados

In [5]:
dataframes_treino = []
dataframes_teste = []
scalers = {}

for campus, dados in df_regressao.groupby("CAMPUS"):
    scaler = MinMaxScaler()
    dados["CAMPUS"] = campus
    dados[["CONSUMO"]] = scaler.fit_transform(dados[["CONSUMO"]])

    for i in range(1, 12 + 1):
        lag = dados['CONSUMO'].shift(i)
        dados[f'LAG_' + '{:02d}'.format(i)] = lag
    dados.dropna(inplace=True)

    treino, teste = train_test_split(dados, test_size=12, random_state=SEED, shuffle=False)

    dataframes_treino.append(treino)
    dataframes_teste.append(teste)
    scalers[campus] = scaler

df_treino = pd.DataFrame(pandas.concat(dataframes_treino, ignore_index=True).sort_values("CAMPUS").sort_values("DATA"))
df_teste = pd.DataFrame(pandas.concat(dataframes_teste, ignore_index=True).sort_values("CAMPUS").sort_values("DATA"))
features = df_treino.drop(df_treino.drop(df_features, axis=1).columns.to_list(), axis=1).columns

df_treino.to_pandas()

,CONSUMO,DATA,TEMP_MIN_MÉD_MENS,TEMP_MÉD_MIN_MENS,TEMP_MÉD_MÉD_MENS,TEMP_MÉD_MAX_MENS,TEMP_MÉD_ACC_MENS,TEMP_MAX_MÉD_MENS,PRECIPITAÇÃO_MÉD_MENS,TEMP_MIN_MIN_MENS,...,LAG_03,LAG_04,LAG_05,LAG_06,LAG_07,LAG_08,LAG_09,LAG_10,LAG_11,LAG_12
995,0.477022,2016-01-31,17.0,22.0,24.0,29.0,758.0,36.0,6.0,17.0,...,0.441629,0.323708,0.242163,0.264466,0.464972,0.552163,0.627135,0.914382,0.658708,0.383904
996,0.533736,2016-02-29,19.0,23.0,26.0,30.0,745.0,36.0,13.0,19.0,...,0.498820,0.441629,0.323708,0.242163,0.264466,0.464972,0.552163,0.627135,0.914382,0.658708
1089,0.444332,2016-02-29,21.0,23.0,27.0,29.0,782.0,34.0,4.0,21.0,...,0.529831,0.566768,0.441407,0.363708,0.269847,0.322046,0.394645,0.452619,0.582030,0.325383
997,0.787135,2016-03-31,18.0,21.0,24.0,27.0,738.0,31.0,3.0,18.0,...,0.522247,0.498820,0.441629,0.323708,0.242163,0.264466,0.464972,0.552163,0.627135,0.914382
1090,0.751678,2016-03-31,17.0,22.0,25.0,29.0,787.0,35.0,3.0,17.0,...,0.408670,0.529831,0.566768,0.441407,0.363708,0.269847,0.322046,0.394645,0.452619,0.582030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
944,0.484074,2023-10-31,15.0,20.0,24.0,29.0,755.0,36.0,15.0,15.0,...,0.030501,0.100169,0.194485,0.218346,0.431739,0.239730,0.190996,0.194485,0.174451,0.149128
80,0.624135,2023-10-31,11.0,17.0,23.0,29.0,708.0,36.0,9.0,11.0,...,0.192079,0.318006,0.402022,0.429896,0.783874,0.420769,0.099279,0.363989,0.368062,0.351818
209,1.000000,2023-10-31,8.0,13.0,21.0,27.0,640.0,33.0,15.0,8.0,...,0.445285,0.706495,0.776246,0.621085,0.787278,0.483719,0.334964,0.301868,0.646708,0.607206
607,0.904992,2023-10-31,14.0,20.0,25.0,30.0,781.0,36.0,11.0,14.0,...,0.292037,0.399596,0.442869,0.505189,0.892156,0.487797,0.330547,0.609747,0.567302,0.510262


## Treino com todos os campi

In [7]:
medidas_todos = pandas.DataFrame()
dataframes_todos = {}

for nome_modelo in modelos:

    for campus, dados_teste in df_teste.groupby("CAMPUS"):
        modelo = treino_split_regr(get_modelo(nome_modelo), df_treino, features)

        dados_treino = df_treino[df_treino["CAMPUS"] == campus]
        medida, df = teste_split_regr(modelo, dados_treino, dados_teste, features, horizonte)
        dataframes_todos[(campus, nome_modelo)] = df

        medidas_todos = pandas.concat([medidas_todos, pandas.DataFrame([[campus, medida["MAPE"], medida["RMSE"]]],
                                                                       columns=["CAMPUS", f"MAPE {nome_modelo}",
                                                                                f"RMSE {nome_modelo}"])])

    medidas_todos = pandas.concat(
        [medidas_todos, pandas.DataFrame(
            [["TOTAL", medidas_todos[f"MAPE {nome_modelo}"].mean(), medidas_todos[f"RMSE {nome_modelo}"].mean()]],
            columns=["CAMPUS", f"MAPE {nome_modelo}", f"RMSE {nome_modelo}"])]).round(5)

medidas_todos

ValueError: invalid literal for int() with base 10: '441,0'



## Treino com campi individuais

In [28]:
medidas_ind = pandas.DataFrame()
dataframes_ind = {}

for nome_modelo in modelos:
    for campus, dados_teste in df_teste.groupby("CAMPUS"):
        dados_treino = df_treino[df_treino["CAMPUS"] == campus]

        modelo = treino_split_regr(get_modelo(nome_modelo), dados_treino, features)

        medida, df = teste_split_regr(modelo, dados_treino, dados_teste, features, horizonte)
        dataframes_ind[(campus, nome_modelo)] = df

        medidas_ind = pandas.concat([medidas_ind, pandas.DataFrame([[campus, medida["MAPE"], medida["RMSE"]]],
                                                                   columns=["CAMPUS", f"MAPE {nome_modelo}",
                                                                            f"RMSE {nome_modelo}"])])

    medidas_ind = pandas.concat(
        [medidas_ind, pandas.DataFrame(
            [["TOTAL", medidas_ind[f"MAPE {nome_modelo}"].mean(), medidas_ind[f"RMSE {nome_modelo}"].mean()]],
            columns=["CAMPUS", f"MAPE {nome_modelo}", f"RMSE {nome_modelo}"])]).round(5)
medidas_ind

/tmp/ipykernel_10163/4223776346.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores = pandas.concat([scores, medidas])
/tmp/ipykernel_10163/4223776346.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores = pandas.concat([scores, medidas])
/tmp/ipykernel_10163/4223776346.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes.

,CAMPUS,MAPE RF,RMSE RF
0,ASSIS CHATEAUBRIAND,0.53241,4305.62504
0,ASTORGA,0.22638,1685.45336
0,BARRACÃO,0.22411,865.39815
0,CAMPO LARGO,0.41915,2925.30782
0,CAPANEMA,0.45857,2153.10752
0,CASCAVEL,0.31085,3484.52435
0,CORONEL VIVIDA,0.34247,857.58211
0,CURITIBA,0.29039,5012.31944
0,FOZ DO IGUAÇU,0.52453,8328.07571
0,GOIOERÊ,0.46499,3239.42925


# Gráficos

## Treino com todos os campi

In [30]:
for campus, dado_real in df_regressao.groupby("CAMPUS"):
    valores_previstos = {}
    dado_real["DATA"] = pandas.to_datetime(dado_real["DATA"])
    dado_real = dado_real.set_index("DATA")
    scaler = scalers[campus]

    for nome_modelo in modelos:
        dados = dataframes_todos[(campus, nome_modelo)].to_pandas()
        dados.index = pandas.to_datetime(dados.index)
        colunas = dados[features].drop(dados.filter(like='LAG_').columns.tolist(), axis=1).columns
        colunas = np.append('CONSUMO', colunas)

        dados[colunas] = scaler.inverse_transform(dados[colunas]).round(2)
        valores_previstos[nome_modelo] = dados

    dado_real = dado_real[12:len(valores_previstos[modelos[0]])+12]

    plt = ts_comparacao(campus, dado_real, valores_previstos)
    plt.savefig(f"resultados/previsoes - treino conjunto/Previsao {horizonte}M {campus}.png", bbox_inches='tight')



KeyError: 'ASSIS CHATEAUBRIAND'

## Treino com campi individuais

In [ ]:
for campus, dado_real in df_regressao.groupby("CAMPUS"):
    valores_previstos = {}
    dado_real["DATA"] = pandas.to_datetime(dado_real["DATA"])
    dado_real = dado_real.set_index("DATA")
    scaler = scalers[campus]

    for nome_modelo in modelos:
        dados = dataframes_ind[(campus, nome_modelo)].to_pandas()
        dados.index = pandas.to_datetime(dados.index)
        colunas = dados[features].drop(dados.filter(like='LAG_').columns.tolist(), axis=1).columns
        colunas = np.append('CONSUMO', colunas)

        dados[colunas] = scaler.inverse_transform(dados[colunas]).round(2)
        valores_previstos[nome_modelo] = dados

    dado_real = dado_real[12:len(valores_previstos[modelos[0]])+12]

    plt = ts_comparacao(campus, dado_real, valores_previstos)
    plt.savefig(f"resultados/previsoes - treino individual/Previsao {horizonte}M {campus}.png", bbox_inches='tight')



In [ ]:
dados

In [ ]:
dado_real